# NIVEAUX DE VULNERABILITE 

**Périmètre des résultats :** les indicateurs sont calculés et présentés séparément,
puis récapitulés par sous-groupe. Aucun score global intersectoriel n'est calculé :
sa pondération reste à décider.

## DEMOGRAPHIQUE (composition du ménage)
   - Méthode de calcul : 
	     * La structure de la population du ménage par groupes d'âge vulnérables (Vulnerable Age Groups / Demographics Breakdown)
			 * Le Ratio de Dépendance Démographique (Age Dependency Ratio)
			 * FHH (Female-Headed Households / Ménages dirigés par des femmes)
			 * HHwDs (Households with Persons with Disabilities / Ménages avec des personnes en situation de handicap)
			 * Ménages à charge de morbidité critique (adultincap ou malnut)

   - Seuil de sévérité : 
	 
	 Pour documenter la vulnérabilité structurelle et la capacité d'adaptation des ménages, la méthodologie repose sur l'operationalisation d'indicateurs normalisés par les agences humanitaires et alignés sur le cadre analytique mondial du Joint and Intersectoral Analysis Framework (JIAF 2.0). 
	 
	 Premièrement, la taille du ménage et le ratio de dépendance démographique sont calculés en distinguant la population inactive (enfants de moins de 18 ans et personnes âgées de 60 ans et plus) de la population active (adultes de 18 à 59 ans). Conformément aux standards de la FAO et des enquêtes multisectorielles de besoins (MSNA), un ratio de dépendance supérieur à 2 est retenu comme seuil de sévérité critique, matérialisant un fardeau économique excessif pour les actifs du foyer. 
	 
	 Deuxiemement, les profils de vulnérabilité spécifique (présence de femmes enceintes ou allaitantes, cas de malnutrition infantile, ou incapacité de travail des adultes) sont transformés en indicateurs binaires contextuels (drapeaux d'alerte). 
	 
	 L'agrégation de ces dimensions permet de constituer un score synthétique de vulnérabilité structurelle et de protection, servant de variable cible ou de co-variable explicative pour les modèles de Machine Learning.
	 
   - Source : 
	     * https://humanitarian.atlassian.net/wiki/spaces/hpc/pages/3993829401/JIAF+Manuals+and+Tool
			 * https://www.fao.org/in-action/drought-portal/preparedness/vulnerability-and-impact-assessment/compendium---vulnerability-indicators/age-dependency-ratio/en

In [ ]:
from pathlib import Path
import inspect
import sys
from IPython.display import Markdown, display

racines = (Path.cwd(), *Path.cwd().parents)
racine_projet = next(
    (racine for racine in racines if (racine / "src" / "tech_scores.py").is_file()),
    None,
)
if racine_projet is None:
    raise FileNotFoundError("Le module src/tech_scores.py est introuvable depuis le dossier courant.")
if str(racine_projet) not in sys.path:
    sys.path.insert(0, str(racine_projet))

from src.tech_scores import (
    calculer_handicap_washington_group,
    calculer_hhs,
    calculer_score_abri_partiel,
    calculer_score_ame_officiel,
    calculer_scores,
    indicateur_oui_non,
    normaliser_texte,
    pipeline_agro_severite_complet,
    pipeline_demographie_et_handicap,
    pipeline_indicateurs_economiques,
    pipeline_vulnerabilite_sociale_decisionnelle,
    pipeline_wash_severite_complet,
    trouver_score_par_intervalle,
)


def afficher_implementation(fonctions):
    for fonction in fonctions:
        display(Markdown(f"```python\n{inspect.getsource(fonction)}\n```"))


afficher_implementation((normaliser_texte, indicateur_oui_non, pipeline_demographie_et_handicap))

## HANDICAP
   - Méthode de calcul : Washington group ->
   - Seuil de sévérité : Niveau Aucun : L'individu a répondu code 1 (« aucune difficulté ») à tous les domaines, Niveau Modéré : L'individu n'a pas de code 3 ou 4, mais a au moins un domaine avec le code 2 (« un peu de difficulté »)., Niveau Sévère : L'individu a au moins un domaine avec le code 3 ou 4.
   - Source : https://www.washingtongroup-disability.com/

**Codage propre à l'enquête :** les six réponses WG-SS sont stockées sous forme de
libellés textuels, pas de codes numériques. La cellule recode explicitement les
réponses connues en niveaux 1 à 4. Les refus et réponses inconnues restent manquants.

In [ ]:
afficher_implementation((calculer_handicap_washington_group,))

## LOGEMENT, ABRI, STATUT D'OCCUPATION

- **Méthode officielle :** pour chaque critère, multiplier les notes de ses variables constitutives. Additionner ensuite les notes des sept critères et diviser par 7. Le score complet va de 1 (vulnérabilité maximale) à 10 (vulnérabilité minimale).
- **Critère A — situation humanitaire face à l'abri :** note `status` × note de `observation_0`, utilisée ici comme proxy de l'état de l'abri. Dans cette enquête, la question réelle est « l'environnement est-il adapté aux personnes qui y vivent ? » : `non` reçoit 1 (situation moins adaptée, donc plus vulnérable), `oui` reçoit 5 (situation adaptée), et « je ne sais pas » reste non renseigné. **Ce proxy ne mesure pas si l'abri est endommagé et ne doit pas être présenté comme la mesure officielle du dommage.**
- **Critère C — lieu d'habitation :** note `herber` × note `environ`, selon les barèmes des modalités du questionnaire et du notebook.
- **Critères B, D, E, F et G :** les champs nécessaires (promiscuité détaillée, toiture, murs, portes/fenêtres et latrines) ne sont pas présents dans la base actuelle. Le notebook ne fabrique pas ces notes. Les colonnes supplémentaires des tables SQL reliées sont incluses automatiquement sous la forme `table__colonne` ; la configuration `BAREMES_ABRI_OPTIONNELS` permet ensuite de saisir les variables réelles et leurs barèmes officiels pour calculer ces critères.
- **Score card réduit :** moyenne des notes de critères calculables parmi A à G ; avec les données actuelles, seuls A et C sont calculables au maximum. Le nombre de critères utilisés est affiché pour chaque ménage. Il suit la logique de la moyenne officielle, mais son dénominateur est le nombre de critères disponibles et il ne remplace pas le score complet.
- **Score card complet sur 7 critères :** moyenne divisée par 7, produite seulement si les sept notes A–G sont présentes pour le ménage ; sinon elle reste non renseignée. Comme A utilise `observation_0` en remplacement de la question officielle sur les dommages, ce résultat serait un score adapté à l'enquête, pas un score strictement identique à l'outil officiel.

Source : [Score card du Cluster Abris RDC](https://www.sheltercluster.org/republique-democratique-du-congo/documents/score-card-du-cluster-abris).

In [ ]:
afficher_implementation((calculer_score_abri_partiel,))

## SECAL 
   - Méthode de calcul : calcul du score HHS -> 
   - Seuil de sévérité : "Peu ou pas de faim (0-1)", "Faim modérée (2-3)", "Faim sévère (4-6)"
   - Source : https://vamresources.manuals.wfp.org/docs/household-hunger-scale-hhs

In [ ]:
afficher_implementation((calculer_hhs,))

## AME/NFI
   - Méthode de calcul : Score card AME -> 
   - Seuil de sévérité : Vulnérabilité faible ou modérée : score de <1 à 2.9, Vulnérabilité aigüe modérée : score de 3 – 3.9 (phase d’urgence), Vulnérabilité aigüe sévère : score de 4-5 (phase de crise)
   - Source :  [reach_shelter_cluster_assessment_scorecardame_final_08122016_to_publish.pdf](https://www.sheltercluster.org/working-group-nfi-practices/documents/revue-des-outils-du-cluster-ame-evaluation-de-loutil-score)

In [ ]:
afficher_implementation((trouver_score_par_intervalle, calculer_score_ame_officiel))

## GENRE
   - Méthode de calcul : *L'Indice de vulnérabilité sociodémographique et décisionnelle
   - Seuil de sévérité : l'indicateur de vulnérabilité critique est activé lorsque le statut matrimonial indique un isolement (veuf/veuve, divorcé/séparé ou célibataire) ET que la décision d'utilisation de l'argent est attribuée à une personne tierce du ménage. Cette opérationnalisation est un proxy contextuel à valider ; les décisions prises par l'homme, la femme ou les deux sont distinguées dans le tableau.
   - Source : https://interagencystandingcommittee.org/

In [ ]:
afficher_implementation((pipeline_vulnerabilite_sociale_decisionnelle,))

## ECONOMIQUE (revenus/dette)
   - Méthode de calcul : *Ratio d'Endettement / Pression Financière *Surcharge locative 
   - Seuil de sévérité : Pour évaluer la vulnérabilité socio-économique et la capacité d'absorption financière des ménages face aux chocs, la méthodologie opérationnalise deux indicateurs de pression monétaire alignés sur les standards des évaluations multisectorielles de besoins (MSNA) et du cadre intersectoriel JIAF 2.0. D'une part, la surcharge locative (Housing Cost Overburden) est mesurée en rapportant la charge mensuelle du loyer au revenu total du foyer. Conformément aux seuils normatifs des clusters Abris et Habitat en contexte de crise, un ménage est classé en situation de vulnérabilité critique lorsque le loyer excède 40% de ses revenus, traduisant une distorsion budgétaire majeure exposant le foyer au risque d'expulsion. D'autre part, la pression de l'endettement est évaluée par le ratio entre le montant global des dettes contractées (en nature ou en espèces) et le revenu mensuel moyen. Un seuil de sévérité critique est établi lorsque l'endettement dépasse l'équivalent de trois mois de revenus, marquant l'épuisement des mécanismes traditionnels d'emprunt social et le basculement vers des stratégies d'adaptation d'urgence.
   - Source : https://interagencystandingcommittee.org/operational-policy-and-advocacy-group/iasc-technical-manual-joint-and-intersectoral-analysis-framework-jiaf-20

In [ ]:
afficher_implementation((pipeline_indicateurs_economiques,))

## WASH (accès eau potable)
   - Méthode de calcul : *Indicateur de source d'approvisionnement précaire *Capacité totale de stockage domestique *Séparation de l'eau de boisson
   - Règle de classement demandée : une source d'eau non améliorée ou l'eau de surface entraîne directement le niveau 3 (critique), même si le stockage et la séparation sont adéquats. « Puits traditionnel » est classé séparément, sans être assimilé à une source améliorée ou non améliorée : son alerte eau et le score WASH restent non calculés en l'absence de cette classification. Sinon, le score est la somme des trois alertes : 3 = critique, 2 = tension, 0–1 = vulnérabilité faible ou modérée.
   - Source : https://www.washcluster.net/

In [ ]:
afficher_implementation((pipeline_wash_severite_complet,))

## AGRO : 
   - Méthode de calcul : *Indice de précarité du capital productif, *Indicateur de vulnérabilité de la main-d'œuvre agricole, *Indicateur de déficit d'irrigation et d'accès à l'eau agricole, *Indicateur d'isolement social et de déficit de capital associatif *Indicateur de vulnérabilité par l'inexpérience technique
   - Seuil de sévérité : Pour évaluer la vulnérabilité agricole et l'insécurité des moyens d'existence, la méthodologie s'appuie sur un indice composite agrégeant cinq dimensions normatives inspirées des cadres JIAF 2.0 et FAO RIMA. Pour chaque ménage déclarant pratiquer l'agriculture, le modèle calcule la précarité du capital productif (via l'accès à la terre terre et aux outils), la vulnérabilité de la force de travail, le déficit d'irrigation, l'isolement associatif, ainsi que l'inexpérience technique. L'addition de ces sous-indicateurs binaires génère un score global de sévérité s'échelonnant de 0 à 5. Le seuil de la zone critique, correspondant à la Phase 3+ (« Sévère ») de l'échelle humanitaire, est officiellement atteint dès qu'un ménage cumule au moins deux facteurs de privation simultanés, ce qui matérialise une rupture structurelle de ses capacités productives et le qualifie comme cible prioritaire d'intervention d'urgence.   
   - Source : https://interagencystandingcommittee.org/operational-policy-and-advocacy-group/iasc-technical-manual-joint-and-intersectoral-analysis-framework-jiaf-20

In [ ]:
afficher_implementation((pipeline_agro_severite_complet,))

## Synthèse des calculs, scores et niveaux par indicateur

Ce tableau récapitule les règles effectivement codées dans les cellules précédentes. Les niveaux sont sectoriels et ne sont pas additionnés en un score global intersectoriel. Les scores binaires ou les seuils marqués « proxy » sont des adaptations à l'enquête, et non des barèmes officiels validés.

| Indicateur | Calcul / score appliqué | Niveaux ou interprétation | Sens et limites |
|---|---|---|---|
| **Démographie** | Taille du ménage calculée par somme des colonnes d'âge et de sexe, si toutes les colonnes requises sont disponibles. Ratio de dépendance = personnes classées à charge / personnes de 18–59 ans actives. Les alertes grossesse, allaitement, adulte incapable, handicap déclaré et malnutrition déclarée restent des indicateurs séparés. | Ratio **> 2 : critique** ; ratio **≤ 2 : seuil critique non atteint**. Les alertes déclaratives sont codées 0/1, sans niveau composite. | Pas de score démographique global. **À vérifier avant interprétation :** plusieurs colonnes d'âge utilisées semblent couvrir des tranches qui se chevauchent (ex. 0–5 et 5–14) ; confirmer leur définition dans le questionnaire pour éviter un double comptage dans la taille et le ratio. |
| **Handicap (WG-SS)** | Six domaines recodés en codes 1 à 4. `score_handicap_wg` = **1** si au moins un domaine a le code 3 ou 4 ; **0** si des réponses sont disponibles mais aucune n'atteint 3 ; manquant si aucun domaine n'est renseigné. | **Aucune difficulté** : tous les domaines au code 1 ; **modéré** : au moins un code 2 et aucun code 3/4 ; **sévère** : au moins un code 3/4. Les réponses partielles non classables sont signalées incomplètes. | Le binaire indique uniquement la sévérité WG-SS ; il ne mesure pas toutes les formes de vulnérabilité du ménage. |
| **Abri — critère A** | Note de `status` × note de `observation_0` (proxy d'adaptation de l'environnement : non = 1, oui = 5). Les notes de statut codées sont : déplacé/réfugié = 1 ; retourné = 1,5 ; non déplacé autochtone hébergé par la famille = 1,75 ; non déplacé autochtone hors famille d'accueil = 2. | Critère individuel : produit des notes, sur l'échelle du score card. | `observation_0` ne mesure pas les dommages de l'abri : c'est un **proxy**, pas la variable officielle du critère A. |
| **Abri — critère C** | Note du lieu d'habitation × note de l'environnement. Notes du lieu : famille d'accueil 2,5 ; centre collectif 1,5 ; belle étoile 1 ; site spontané 1,75 ; occupation gratuite 2 ; colocataire en maison louée 3 ; locataire 3,5 ; propre parcelle 4. Notes d'environnement : dangereux avec promiscuité <1,5 m² = 1 ; dangereux = 1,5 ; non dangereux avec promiscuité <1,5 m² = 1,75 ; non dangereux = 2,5. | Critère individuel : produit des notes, sur l'échelle du score card. | Modalités et barèmes proviennent du codage présent dans le notebook ; les facteurs manquants restent non calculés. |
| **Abri — score card réduit** | Moyenne des notes des critères calculables parmi A–G ; actuellement A et/ou C au maximum. | **1 à 10** : selon la méthode du score card, une note basse signale davantage de vulnérabilité et une note haute moins de vulnérabilité. | Score partiel exploratoire : dénominateur égal au nombre de critères disponibles, pas 7 ; afficher le nombre de critères utilisés et ne pas le confondre avec le score complet. |
| **Abri — score complet adapté** | Somme des notes A–G / **7**, seulement si les sept critères sont calculables. Les critères B, D, E, F et G attendent leurs variables et barèmes officiels. | **1 à 10** : 1 = vulnérabilité maximale ; 10 = vulnérabilité minimale. | Reste manquant si un critère manque ; même complet, le résultat reste adapté à l'enquête tant que A utilise `observation_0` en proxy. |
| **SECAL (HHS)** | Trois questions HHS : non = 0 ; oui avec fréquence ≤10 = 1 ; oui avec fréquence >10 = 2. Somme uniquement si les trois scores sont renseignés. | **0–1 :** peu ou pas de faim ; **2–3 :** faim modérée ; **4–6 :** faim sévère (cible sévère = score ≥4). | Score élevé = faim plus sévère. Les réponses/fréquences manquantes ne sont pas imputées. |
| **AME/NFI** | Chaque article reçoit son score selon le barème AME codé (**0 à 5**, score élevé = privation plus importante). Le score global est la moyenne non pondérée des articles disponibles. Bidons, supports de couchage et couvertures sont d'abord rapportés à la taille du ménage ; les autres quantités sont utilisées telles quelles. | **<3 :** vulnérabilité faible/modérée ; **3 à <4 :** aiguë modérée / urgence ; **4 à 5 :** aiguë sévère / crise. | Les seuils d'articles diffèrent selon le bien et sont définis dans `BAREME_AME`. Le nombre d'articles effectivement scorés est conservé ; la moyenne porte sur les articles disponibles. |
| **Genre — proxy décisionnel** | Score binaire = isolement matrimonial (veuf/veuve, divorcé/séparé ou célibataire) **ET** décision d'utilisation de l'argent attribuée à une personne tierce du ménage. | **1 :** les deux conditions réunies (proxy de vulnérabilité critique) ; **0 :** au moins une condition absente ; manquant si statut ou décision inconnu. | Proxy contextuel à valider ; les modalités homme, femme ou décision conjointe ne sont pas assimilées à une décision tierce. |
| **Économique** | Ratios distincts : loyer de déplacement / revenu positif ; dette / revenu positif ; loyer reçu / revenu positif (descriptif). Des indicateurs 0/1 sont dérivés pour les alertes. | **Surcharge locative critique :** loyer / revenu >0,40. **Endettement critique :** dette / revenu >3. Le revenu locatif reste descriptif ; dette positive est aussi signalée séparément. | Pas de score économique combiné ni de niveau pour les ratios sous les seuils. Un revenu nul/négatif empêche le calcul du ratio ; les montants inconnus restent manquants. |\n
| **WASH** | Somme de trois alertes (1 point chacune) : source non améliorée/eau de surface ; stockage <15 L/personne ; absence de séparation de l'eau de boisson. Une source confirmée non améliorée ou eau de surface force directement le score à **3**. | **0–1 :** vulnérabilité faible/modérée ; **2 :** tension ; **3 :** critique. | Les trois composantes doivent être connues pour la somme. Le **puits traditionnel** est classé à part : sans classification établie, score et niveau restent manquants. |
| **AGRO** | Pour les ménages déclarant pratiquer l'agriculture, somme de cinq alertes binaires : précarité productive (terre **ou** outils manquants), aucune main-d'œuvre, absence de point d'eau agricole, absence d'association, moins de 2 ans d'expérience. | **0–5** ; vulnérabilité agricole sévère si **score ≥2**. | Calculé uniquement si le ménage pratique l'agriculture et si les cinq composantes sont renseignées ; score élevé = cumul de facteurs de vulnérabilité. |

**Lecture transversale :** les échelles n'ont pas toutes le même sens. Pour l'abri, la note officielle augmente lorsque la vulnérabilité diminue ; pour HHS, AME et AGRO, un score plus élevé indique davantage de difficultés. Les indicateurs démographiques et économiques sont conservés séparément. Les seuils adaptés ou les proxies doivent être validés avec les sources méthodologiques et l'équipe projet avant de servir de vérité terrain pour un modèle prédictif.


## Chargement SQL, calcul et tableaux de résultats

**Ordre d'exécution :** avant de lancer la cellule SQL ci-dessous, exécute les cellules de code précédentes qui définissent les fonctions de calcul, ou choisis **Exécuter tout** depuis le début du notebook. Cette section est la seule qui ouvre la base SQL.

La cellule finale ci-dessous lit la base existante sans la modifier, valide les
colonnes utilisées et la cardinalité des jointures par `id_menage`, puis calcule les
indicateurs. Elle affiche le détail par indicateur et un tableau par sous-groupe
(SECAL, AME/NFI, WASH, AGRO, etc.). Les seuils non définis restent non classés.
Aucun score global intersectoriel n'est calculé.


In [ ]:
from pathlib import Path
from html import escape
import sqlite3
import pandas as pd
from IPython.display import HTML, display

# Recherche de la base existante ; aucune écriture SQL n'est effectuée.
racines = (Path.cwd(), *Path.cwd().parents)
chemin_base = next((racine / "outputs" / "enquete_ms.db" for racine in racines if (racine / "outputs" / "enquete_ms.db").is_file()), None)
if chemin_base is None:
    raise FileNotFoundError("Base SQLite introuvable : le notebook doit rester dans le projet.")

tables = {
    "META_ENQUETE": {"id_menage", "date_enquete"},
    "MENAGE": {"id_menage", "repondant", "sexe", "statmtr", "decision", "status"},
    "STATUT_MENAGE": {"id_menage", "herber", "environ", "loyer_depl_montant", "loyer_auto_montant"},
    "COMPOSITION_MENAGE": {"id_menage", *[f"{s}_{g}" for s in ("h", "f") for g in ("0_5", "6_24", "25_59", "5_14", "15_17", "18_49", "50_59", "plus60")], "fem_enc", "fem_all"},
    "VULNERABILITE_SANTE": {"id_menage", "vis_ss", "hear_ss", "mob_ss", "cog_ss", "sc_ss", "com_ss", "observation_0", "adultincap", "hand", "malnut"},
    "ECONOMIE_REVENU": {"id_menage", "revenu", "montant_dette"},
    "SECURITE_ALIMENTAIRE": {"id_menage", "ressources", "repas_r", "suffisant", "repas_f", "manque", "repas_m"},
    "BIENS_AME": {"id_menage", "nb_bidon", "nb_cass", "nb_bas", "nb_outil", "nb_couch", "nb_couv", "nb_habit_femme", "nb_habit_e"},
    "WASH_EAU": {"id_menage", "capacite", "capacites", "eau", "separer"},
    "AGRICULTURE": {"id_menage", "agro", "terre", "outims", "personne", "eau_point", "asso", "date_agro"},
}

conn = sqlite3.connect(chemin_base.resolve().as_uri() + "?mode=ro", uri=True)
try:
    tables_db = {row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    if set(tables) - tables_db:
        raise ValueError(f"Tables absentes : {sorted(set(tables) - tables_db)}")
    for table, attendues in tables.items():
        presentes = {row[1] for row in conn.execute(f'PRAGMA table_info("{table}")')}
        if attendues - presentes:
            raise ValueError(f"Colonnes absentes de {table} : {sorted(attendues - presentes)}")
        total, distincts = conn.execute(f'SELECT COUNT(*), COUNT(DISTINCT id_menage) FROM "{table}"').fetchone()
        if total != distincts:
            raise ValueError(f"La clé id_menage n'est pas unique dans {table}.")
    erreurs_fk = conn.execute("PRAGMA foreign_key_check").fetchall()
    if erreurs_fk:
        raise ValueError(f"Clés étrangères orphelines détectées : {erreurs_fk[:10]}")

    requete = """
    SELECT m.id_menage, me.date_enquete,
           m.repondant, m.sexe, m.statmtr, m.decision, m.status,
           sm.herber AS lieu_hab, sm.environ, sm.loyer_depl_montant, sm.loyer_auto_montant,
           cm.h_0_5, cm.h_6_24, cm.h_25_59, cm.h_5_14, cm.h_15_17, cm.h_18_49, cm.h_50_59, cm.h_plus60,
           cm.f_0_5, cm.f_6_24, cm.f_25_59, cm.f_5_14, cm.f_15_17, cm.f_18_49, cm.f_50_59, cm.f_plus60,
           cm.fem_enc, cm.fem_all,
           vs.vis_ss, vs.hear_ss, vs.mob_ss, vs.cog_ss, vs.sc_ss, vs.com_ss,
           vs.observation_0, vs.adultincap, vs.hand, vs.malnut,
           er.revenu, er.montant_dette,
           sa.ressources, sa.repas_r, sa.suffisant, sa.repas_f, sa.manque, sa.repas_m,
           ba.nb_bidon, ba.nb_cass, ba.nb_bas, ba.nb_outil, ba.nb_couch, ba.nb_couv, ba.nb_habit_femme, ba.nb_habit_e,
           we.capacite, we.capacites, we.eau, we.separer,
           ag.agro, ag.terre, ag.outims, ag.personne, ag.eau_point, ag.asso, ag.date_agro
    FROM MENAGE m
    INNER JOIN META_ENQUETE me ON me.id_menage = m.id_menage
    LEFT JOIN STATUT_MENAGE sm ON sm.id_menage = m.id_menage
    LEFT JOIN COMPOSITION_MENAGE cm ON cm.id_menage = m.id_menage
    LEFT JOIN VULNERABILITE_SANTE vs ON vs.id_menage = m.id_menage
    LEFT JOIN ECONOMIE_REVENU er ON er.id_menage = m.id_menage
    LEFT JOIN SECURITE_ALIMENTAIRE sa ON sa.id_menage = m.id_menage
    LEFT JOIN BIENS_AME ba ON ba.id_menage = m.id_menage
    LEFT JOIN WASH_EAU we ON we.id_menage = m.id_menage
    LEFT JOIN AGRICULTURE ag ON ag.id_menage = m.id_menage
    """
    # Inclut les champs ajoutés ultérieurement aux tables liées, sans modifier SQL.
    alias_tables = {
        "META_ENQUETE": "me", "MENAGE": "m", "STATUT_MENAGE": "sm",
        "COMPOSITION_MENAGE": "cm", "VULNERABILITE_SANTE": "vs",
        "ECONOMIE_REVENU": "er", "SECURITE_ALIMENTAIRE": "sa",
        "BIENS_AME": "ba", "WASH_EAU": "we", "AGRICULTURE": "ag",
    }
    colonnes_optionnelles = []
    for table, alias in alias_tables.items():
        for info_colonne in conn.execute(f'PRAGMA table_info("{table}")').fetchall():
            colonne = info_colonne[1]
            if colonne != "id_menage":
                nom_qualifie = f"{table.lower()}__{colonne}"
                colonne_sql = '"' + colonne.replace('"', '""') + '"'
                alias_sql = '"' + nom_qualifie.replace('"', '""') + '"'
                colonnes_optionnelles.append(f"{alias}.{colonne_sql} AS {alias_sql}")
    projection_optionnelle = ",\n           " + ",\n           ".join(colonnes_optionnelles)
    requete = requete.replace(
        "\n    FROM MENAGE m",
        f"{projection_optionnelle}\n    FROM MENAGE m",
        1,
    )
    df_sql = pd.read_sql_query(requete, conn)
    nombre_menages = conn.execute("SELECT COUNT(*) FROM MENAGE").fetchone()[0]
finally:
    conn.close()

if len(df_sql) != nombre_menages or df_sql["id_menage"].nunique() != nombre_menages:
    raise ValueError(f"La jointure n'est pas 1:1 : {len(df_sql)} lignes pour {nombre_menages} ménages.")

# La taille calculée est disponible avant ses usages par les scores AME et WASH.
df_scores = calculer_scores(df_sql)
display(pd.DataFrame.from_dict(df_scores.attrs["diagnostic_indicateurs"], orient="index"))

mesures = [
    ("Démographie", "Taille du ménage", "taille_menage", "Personnes", "Descriptif", "Non pondéré", None),
    ("Démographie", "Ratio de dépendance", "ratio_dependance", "Dépendants / actifs", "Critique si > 2", "Indicateur distinct", None),
    ("Démographie", "Femme enceinte", "cible_femme_enceinte", "Binaire 0/1", "Présence déclarée", "Aucune agrégation", None),
    ("Démographie", "Femme allaitante", "cible_femme_allaitante", "Binaire 0/1", "Présence déclarée", "Aucune agrégation", None),
    ("Démographie", "Adulte incapable", "cible_adulte_incapable", "Binaire 0/1", "Présence déclarée", "Aucune agrégation", None),
    ("Démographie", "Handicap déclaré par le ménage", "cible_handicap_menage_declare", "Binaire 0/1", "Déclaration ménage", "Distinct du WG-SS", None),
    ("Démographie", "Malnutrition déclarée", "cible_malnutrition_declaree", "Binaire 0/1", "Présence déclarée", "Aucune agrégation", None),
    ("Handicap", "WG-SS sévère", "score_handicap_wg", "Binaire 0/1", "Sévérité WG-SS ≥ 3", "Indicateur unique", "categorie_handicap_wg"),
("Abri", "Score card réduit (critères disponibles)", "score_card_abri_reduit", "Échelle 1–10; haut = moins vulnérable", "Moyenne des notes de critères disponibles", "Dénominateur = nombre de critères calculables; proxy A", None),
    ("Abri - détail", "Critère A : statut × adaptation (proxy)", "score_abri_critere_a", "Produit des notes du critère", "Note A; adaptation utilisée comme proxy de dommage", "Multiplication officielle; proxy enquête", None),
    ("Abri - détail", "Critère C : lieu × environnement", "score_abri_critere_c", "Produit des notes du critère", "Note C du barème", "Multiplication officielle", None),
("Abri", "Score complet adapté sur 7 critères", "score_card_abri_officiel", "Échelle 1–10; haut = moins vulnérable", "Renseigné uniquement si A–G sont calculés; A utilise un proxy", "Somme A–G / 7", None),
    ("SECAL", "Score HHS", "score_hhs_total", "0–6", "0–1 faible; 2–3 modéré; 4–6 sévère", "Somme des trois questions", "categorie_hhs"),
    ("AME/NFI", "Score AME global", "score_ame_global", "0–5", "<3 faible/modéré; 3–<4 urgence; ≥4 crise", "Moyenne à poids égaux des articles disponibles", "categorie_ame_phase"),
    ("Genre - détail", "Isolement matrimonial (proxy)", "score_isolement_matrimonial_proxy", "Binaire 0/1", "Proxy contextuel", "Indicateur distinct", None),
    ("Genre - détail", "Décision financière tierce", "cible_exclusion_decisionnelle", "Binaire 0/1", "Tierce personne = 1; autres réponses connues = 0", "Indicateur distinct", None),
    ("Genre", "Isolement + décision tierce", "score_genre_vulnerabilite_critique", "Binaire 0/1", "Critique si les deux conditions sont réunies", "ET logique; proxy contextuel", None),
    ("Économique", "Loyer payé / revenu", "ratio_loyer_revenu", "Ratio", "Critique si > 0,40", "Indicateur distinct", None),
    ("Économique", "Loyer reçu / revenu", "ratio_revenu_locatif", "Ratio de revenu complémentaire", "Descriptif; ne constitue pas une alerte de vulnérabilité", "Indicateur distinct", None),
    ("Économique", "Dette / revenu mensuel", "ratio_dette_revenu", "Ratio", "Critique si > 3", "Indicateur distinct", None),
    ("Économique", "Revenu locatif reçu", "cible_revenu_locatif", "Binaire 0/1", "Descriptif; revenu complémentaire", "Indicateur distinct", None),
    ("WASH", "Score WASH", "score_global_wash", "0–3 alertes", "3 critique; 2 tension; 0–1 faible/modérée; source non améliorée confirmée force 3", "Source non améliorée confirmée prioritaire; sinon somme des alertes; puits traditionnel classé séparément", "niveau_wash"),
    ("AGRO", "Score agricole", "score_global_agro", "0–5", "Critique si ≥ 2", "Somme des cinq facteurs", None),
]
for article in BAREME_AME:
    pond = "Poids égal; quantité / taille du ménage" if article in ARTICLES_PONDERES_PAR_TAILLE else "Poids égal; quantité brute"
    mesures.append(("AME/NFI - détail", f"Score {article}", f"score_{article}", "0–5", "Barème AME", pond, None))
for colonne, nom in [
    ("cible_wash_eau_critique", "Source d'eau à risque"),
    ("cible_wash_stockage_insuffisant", "Stockage < 15 L/personne"),
    ("cible_wash_absence_separation", "Absence de séparation de l'eau"),
    ("cible_precarite_productive", "Manque de terre ou d'outils"),
    ("cible_agro_sans_main_doeuvre", "Aucune main-d'œuvre agricole"),
    ("cible_agro_sans_eau", "Pas de point d'eau agricole"),
    ("cible_agro_sans_association", "Pas d'association"),
    ("cible_agro_novice", "Moins de 2 ans d'expérience agricole"),
]:
    groupe = "WASH - détail" if colonne.startswith("cible_wash") else "AGRO - détail"
    mesures.append((groupe, nom, colonne, "Binaire 0/1", "Composante", "Poids égal dans le sous-groupe", None))

COULEURS_NIVEAU = {
    "faible": "#d9ead3",
    "modere": "#fff2cc",
    "critique": "#f4cccc",
    "neutre": "#e7e6e6",
    "descriptif": "#d9eaf7",
}

def classe_couleur_niveau(libelle):
    texte = normaliser_texte(libelle)
    if any(mot in texte for mot in ("faible", "peu ou pas", "aucune difficulte", "non atteint", "sous seuil", "pas critique")):
        return "faible"
    if any(mot in texte for mot in ("severe", "critique", "crise", "niveau 3")):
        return "critique"
    if any(mot in texte for mot in ("modere", "tension", "urgence")):
        return "modere"
    return "neutre"

def couleur_score_abri(note):
    """Rouge à 1, jaune à 5, vert à 10, conformément au sens du score abri."""
    note = min(10.0, max(1.0, float(note)))
    if note <= 5:
        debut, fin, proportion = (215, 48, 39), (255, 235, 132), (note - 1) / 4
    else:
        debut, fin, proportion = (255, 235, 132), (26, 152, 80), (note - 5) / 5
    rgb = tuple(round(a + (b - a) * proportion) for a, b in zip(debut, fin))
    return "#%02x%02x%02x" % rgb

def pastille_effectif(libelle, nombre, couleur):
    fond = COULEURS_NIVEAU.get(couleur, couleur)
    return (
        f'<span style="display:inline-block;background:{fond};padding:3px 7px;'
        f'margin:2px;border-radius:4px;white-space:nowrap">'
        f'{escape(str(libelle))} : <b>{int(nombre):,}</b></span>'
    )

def construire_effectifs_par_niveau(colonne, valeurs, niveau_col):
    if niveau_col:
        effectifs = df_scores[niveau_col].value_counts(dropna=True, sort=False)
        if effectifs.empty:
            return '<span style="color:#666">Aucun ménage classé</span>'
        return " ".join(
            pastille_effectif(nom, nombre, classe_couleur_niveau(nom))
            for nom, nombre in effectifs.items()
        )

    seuils = {
        "ratio_dependance": (2, "Ratio ≤ 2 : seuil critique non atteint", "Ratio > 2 : critique"),
        "ratio_loyer_revenu": (.4, "Loyer/revenu ≤ 0,40", "Loyer/revenu > 0,40 : critique"),
        "ratio_dette_revenu": (3, "Dette/revenu ≤ 3", "Dette/revenu > 3 : critique"),
    }
    if colonne in seuils:
        seuil, libelle_bas, libelle_haut = seuils[colonne]
        bas, haut = int(valeurs.le(seuil).sum()), int(valeurs.gt(seuil).sum())
        return " ".join((
            pastille_effectif(libelle_bas, bas, "faible"),
            pastille_effectif(libelle_haut, haut, "critique"),
        ))

    if colonne == "ratio_revenu_locatif":
        avec_revenu = int(valeurs.gt(0).sum())
        sans_revenu = int(valeurs.eq(0).sum())
        return " ".join((
            pastille_effectif("Ratio > 0 : revenu locatif déclaré (descriptif)", avec_revenu, "descriptif"),
            pastille_effectif("Ratio = 0", sans_revenu, "neutre"),
            '<span style="color:#555">Aucun seuil critique défini pour cet indicateur.</span>',
        ))

    if colonne == "score_card_abri_reduit":
        morceaux = []
        for note in range(1, 11):
            nombre = int((valeurs.sub(note).abs() < 1e-8).sum())
            morceaux.append(pastille_effectif(f"Score {note}", nombre, couleur_score_abri(note)))
        non_entiers = int((valeurs.notna() & (valeurs.sub(valeurs.round()).abs() >= 1e-8)).sum())
        if non_entiers:
            morceaux.append(pastille_effectif("Scores non entiers", non_entiers, "neutre"))
        return " ".join(morceaux)

    if colonne == "score_global_agro":
        sous_seuil = int(valeurs.lt(2).sum())
        critique = int(valeurs.ge(2).sum())
        return " ".join((
            pastille_effectif("Score 0–1 : seuil critique non atteint", sous_seuil, "faible"),
            pastille_effectif("Score ≥2 : sévère", critique, "critique"),
        ))

    if colonne.startswith("cible_") or colonne in {"score_genre_vulnerabilite_critique", "score_handicap_wg"}:
        sans_alerte = int(valeurs.eq(0).sum())
        avec_alerte = int(valeurs.eq(1).sum())
        return " ".join((
            pastille_effectif("0 : sans alerte", sans_alerte, "faible"),
            pastille_effectif("1 : alerte", avec_alerte, "critique"),
        ))

    if colonne.startswith("score_nb_") or colonne in {"score_hhs_total", "score_ame_global"}:
        return " ".join(
            pastille_effectif(f"Score {note:g}", int((valeurs.sub(note).abs() < 1e-8).sum()), "neutre")
            for note in sorted(valeurs.dropna().unique())
        ) or '<span style="color:#666">Aucun score calculé</span>'

    return '<span style="color:#666">Pas de niveau défini</span>'

lignes = []
for groupe, indicateur, colonne, echelle, seuil, ponderation, niveau_col in mesures:
    valeurs = pd.to_numeric(df_scores[colonne], errors="coerce")
    lignes.append({
        "Sous-groupe": groupe,
        "Indicateur": indicateur,
        "Colonne résultat": colonne,
        "Score moyen / prévalence": valeurs.mean(),
        "Échelle": echelle,
        "Seuil / niveau": seuil,
        "Effectifs par niveau": construire_effectifs_par_niveau(colonne, valeurs, niveau_col),
        "Pondération": ponderation,
        "Ménages calculés": int(valeurs.notna().sum()),
        "Ménages manquants": int(valeurs.isna().sum()),
    })
recapitulatif_indicateurs = pd.DataFrame(lignes)

def recap_groupe(nom, colonne, echelle, ponderation, note, masque=None):
    if colonne is None:
        return {"Sous-groupe": nom, "Score moyen": float("nan"), "Échelle": echelle, "Pondération": ponderation, "Ménages calculés": 0, "Note": note}
    valeurs = pd.to_numeric(df_scores[colonne], errors="coerce")
    if masque is not None:
        valeurs = valeurs.where(masque)
    return {"Sous-groupe": nom, "Score moyen": valeurs.mean(), "Échelle": echelle, "Pondération": ponderation, "Ménages calculés": int(valeurs.notna().sum()), "Note": note}

recapitulatif_sous_groupes = pd.DataFrame([
    recap_groupe("Démographie", None, "Indicateurs distincts", "À décider", "Taille, ratio et alertes ne sont pas fusionnés."),
    recap_groupe("Handicap", "score_handicap_wg", "0/1", "Indicateur unique", "Détail de sévérité dans le tableau indicateur."),
    recap_groupe("Abri - score réduit", "score_card_abri_reduit", "Échelle 1–10", "Moyenne des critères calculables A–G", "Actuellement seuls A et C au maximum; A utilise observation_0 comme proxy."),
    recap_groupe("Abri - score complet adapté", "score_card_abri_officiel", "Échelle 1–10", "Somme des 7 critères / 7", "Non calculé tant que A–G ne sont pas tous mesurables; A reste un proxy."),
    recap_groupe("SECAL", "score_hhs_total", "0–6", "Somme de 3 questions", "Niveaux HHS standard."),
    recap_groupe("AME/NFI", "score_ame_global", "0–5", "Moyenne des articles disponibles", "Trois articles sont rapportés à la taille du ménage."),
    recap_groupe("Genre", "score_genre_vulnerabilite_critique", "Binaire 0/1", "Isolement matrimonial ET décision tierce", "Proxy contextuel à valider; autres modalités décisionnelles conservées."),
    recap_groupe("Économique", None, "Ratios distincts", "À décider", "Loyer/revenu et dette/revenu ne sont pas fusionnés."),
    recap_groupe("WASH", "score_global_wash", "0–3; 3 critique, 2 tension, 0–1 faible/modérée", "Source non améliorée confirmée force 3; sinon somme des alertes", "Les puits traditionnels sont classés à part et n'ont pas de score tant que leur statut n'est pas établi."),
    recap_groupe("AGRO", "score_global_agro", "0–5 facteurs", "Somme des 5 facteurs", "Calculé pour les ménages agricoles seulement.", df_scores["cible_pratique_agro"].eq(1)),
])

colonnes_detail = ["id_menage"] + list(dict.fromkeys(col for _, _, col, _, _, _, _ in mesures))
colonnes_detail += ["categorie_hhs", "categorie_ame_phase", "categorie_handicap_wg", "categorie_source_eau", "niveau_wash"]
detail_indicateurs = df_scores[[col for col in dict.fromkeys(colonnes_detail) if col in df_scores.columns]].copy()

print(f"Base lue sans modification : {chemin_base} | {nombre_menages:,} ménages | jointures 1:1 vérifiées.")
print("Aucun score global intersectoriel n'est calculé.")
display(recapitulatif_sous_groupes)
colonnes_recap = [
    "Sous-groupe", "Indicateur", "Colonne résultat", "Score moyen / prévalence",
    "Échelle", "Seuil / niveau", "Effectifs par niveau", "Pondération",
    "Ménages calculés", "Ménages manquants",
]
recapitulatif_indicateurs = recapitulatif_indicateurs[colonnes_recap]
display(HTML(recapitulatif_indicateurs.to_html(index=False, escape=False)))
print("Détail par ménage (10 premières lignes) :")
detail_apercu = detail_indicateurs.head(10)
if "score_card_abri_reduit" in detail_apercu.columns:
    display(detail_apercu.style.map(
        lambda note: f"background-color: {couleur_score_abri(note)}" if pd.notna(note) else "",
        subset=["score_card_abri_reduit"],
    ))
else:
    display(detail_apercu)